In [0]:
%sql
-- FACT FINANCIALS (from general ledger transactions)
CREATE OR REPLACE TABLE plworkforce_catalog.003_gold.fact_financials AS
SELECT 
    gl.transaction_id,
    gl.posting_date,
    gl.year,
    gl.month,
    gl.year_month,
    gl.company_id,
    gl.account_id,
    gl.department_id,
    gl.transaction_type,
    gl.debit_amount,
    gl.credit_amount,
    gl.amount,
    a.account_type,
    a.category,
    CASE 
        WHEN a.account_type = 'Revenue' THEN gl.credit_amount - gl.debit_amount
        WHEN a.account_type IN ('Expense', 'Cogs') THEN gl.debit_amount - gl.credit_amount
        ELSE gl.amount
    END AS net_amount
FROM plworkforce_catalog.002_silver.general_ledgers gl
INNER JOIN plworkforce_catalog.002_silver.accounts a ON gl.account_id = a.account_id
WHERE gl.status = 'Posted';

-- FACT PAYROLL (compensation details)
-- Note: gross_salary in silver is now calculated as base_salary + bonus + overtime_pay + commission + allowances
CREATE OR REPLACE TABLE plworkforce_catalog.003_gold.fact_payroll AS
SELECT 
    p.payroll_id,
    p.employee_id,
    p.company_id,
    p.department_id,
    p.pay_date,
    p.year,
    p.month,
    p.year_month,
    p.gross_salary,
    p.bonus,
    p.overtime_pay,
    p.commission,
    p.allowances,
    p.gross_salary AS total_compensation,
    p.tax_deduction,
    p.social_security,
    p.health_insurance,
    p.retirement_contribution,
    p.other_deductions,
    p.net_salary,
    e.position,
    e.hire_date,
    e.employee_status
FROM plworkforce_catalog.002_silver.payroll p
INNER JOIN plworkforce_catalog.002_silver.employee e ON p.employee_id = e.employee_id
WHERE p.status = 'PAID';

-- Verify fact tables created
SELECT 'fact_financials' AS table_name, COUNT(*) AS row_count FROM plworkforce_catalog.003_gold.fact_financials
UNION ALL
SELECT 'fact_payroll' AS table_name, COUNT(*) AS row_count FROM plworkforce_catalog.003_gold.fact_payroll;

In [0]:
%sql
-- Sample from fact_financials
SELECT * FROM plworkforce_catalog.003_gold.fact_financials
ORDER BY posting_date DESC, transaction_id
LIMIT 10;

-- Sample from fact_payroll
SELECT * FROM plworkforce_catalog.003_gold.fact_payroll
ORDER BY pay_date DESC, payroll_id
LIMIT 10;

-- Summary statistics
SELECT 
    'fact_financials' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT company_id) AS companies,
    COUNT(DISTINCT account_id) AS accounts,
    COUNT(DISTINCT year_month) AS months,
    MIN(posting_date) AS earliest_date,
    MAX(posting_date) AS latest_date
FROM plworkforce_catalog.003_gold.fact_financials

UNION ALL

SELECT 
    'fact_payroll' AS table_name,
    COUNT(*) AS total_rows,
    COUNT(DISTINCT company_id) AS companies,
    COUNT(DISTINCT employee_id) AS employees,
    COUNT(DISTINCT year_month) AS months,
    MIN(pay_date) AS earliest_date,
    MAX(pay_date) AS latest_date
FROM plworkforce_catalog.003_gold.fact_payroll;